In [ ]:
import os
import zipfile
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
ZIP_PATH = '/content/dataset.zip'
EXTRACT_PATH = '/content/dataset'
if not os.path.exists(EXTRACT_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z: z.extractall(EXTRACT_PATH)

In [ ]:
train_ds_vgg = tf.keras.utils.image_dataset_from_directory(
    EXTRACT_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

In [ ]:
val_ds_vgg = tf.keras.utils.image_dataset_from_directory(
    EXTRACT_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

In [ ]:
vgg_base = tf.keras.applications.VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
vgg_base.trainable = False

In [ ]:
vgg_model = models.Sequential([
    vgg_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(len(train_ds_vgg.class_names), activation='softmax')
])

In [ ]:
vgg_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
vgg_model.fit(train_ds_vgg.prefetch(tf.data.AUTOTUNE), validation_data=val_ds_vgg, epochs=10)
vgg_model.save('vgg16_model.keras')

In [ ]:
def map_ae(img, label):
    img = img / 255.0
    return img, img

ae_ds = tf.keras.utils.image_dataset_from_directory(
    EXTRACT_PATH,
    image_size=(512, 512),
    batch_size=16,
    label_mode='categorical'
).map(map_ae).prefetch(tf.data.AUTOTUNE)

In [ ]:
input_img = layers.Input(shape=(512, 512, 3))
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = layers.MaxPooling2D((2, 2), padding='same')(x)
x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
encoded = layers.MaxPooling2D((2, 2), padding='same')(x)

In [ ]:
x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(encoded)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = layers.UpSampling2D((2, 2))(x)
decoded = layers.Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

In [ ]:
autoencoder = models.Model(input_img, decoded)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(ae_ds, epochs=10)
autoencoder.save('autoencoder_model.keras')